In [1]:
import gradio as gr
import pandas as pd
import torch
import torch.nn as nn
import json
import os
from catboost import CatBoostClassifier

# --- 1. АРХИТЕКТУРА МОДЕЛИ BERT (TRANSFORMER) ---
class TransformerRec(nn.Module):
    def __init__(self, vocab_size, hidden=256, heads=8, layers=4, max_len=20, dropout=0.2):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, hidden)
        self.pos_emb = nn.Embedding(max_len, hidden)
        self.dropout = nn.Dropout(dropout)
        encoder_layer = nn.TransformerEncoderLayer(d_model=hidden, nhead=heads, dropout=dropout, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=layers)
        self.fc = nn.Linear(hidden, vocab_size)

    def forward(self, x):
        positions = torch.arange(x.size(1), device=x.device).unsqueeze(0)
        x = self.dropout(self.token_emb(x) + self.pos_emb(positions))
        x = self.transformer(x)
        return self.fc(x)

# --- 2. НАСТРОЙКИ И ЗАГРУЗКА ДАННЫХ ---
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
data_folder = "../gym_model"

LEVEL_MAP = {'Novice': 0, 'Beginner': 1, 'Intermediate': 2, 'Advanced': 3}
# Маппинг для приведения любого "легкого" инвентаря к обученному классу
INVENTORY_MAP = {
    'Bands': 'Bodyweight',
    'Cardio': 'Bodyweight',
    'Bodyweight': 'Bodyweight',
    'Machine': 'Machine',
    'Dumbbell': 'Dumbbell',
    'Barbell': 'Barbell'
}

with open(f'{data_folder}/vocab.json', 'r') as f:
    vocab = json.load(f)
id2name = {v: k for k, v in vocab.items()}

df_meta = pd.read_csv("../programs_detailed_boostcamp_kaggle.csv")
exercise_meta_dict = df_meta.drop_duplicates(subset=['exercise_name']).set_index('exercise_name').to_dict('index')

df_equip = pd.read_csv("./datasets/id_to_equipment_mapping_FIXED.csv")
id_to_eq_dict = df_equip.set_index('candidate_id')['equipment_golden'].to_dict()

model = TransformerRec(vocab_size=len(vocab)).to(device)
bert_path = f'{data_folder}/checkpoints-2/gym_bert_v2_ep27_hit0.3522.pth'
if os.path.exists(bert_path):
    model.load_state_dict(torch.load(bert_path, map_location=device))
    model.eval()

cat_ranker = CatBoostClassifier()
cat_path = "catboost_recommender_final.cbm"
if not os.path.exists(cat_path):
    cat_path = f'{data_folder}/{cat_path}'

if os.path.exists(cat_path):
    cat_ranker.load_model(cat_path)
    print(f"✅ CatBoost загружен: {cat_path}")

# --- 3. ЛОГИКА ПРЕДСКАЗАНИЯ ---
def predict_debug(history_ids, user_profile, top_k=10):
    input_seq = torch.LongTensor([history_ids]).to(device)
    with torch.no_grad():
        logits = model(input_seq)
        probs = torch.softmax(logits[0, -1, :], dim=-1)
    
    scores, indices = torch.topk(probs, k=100) 
    candidates = []
    
    last_ex_name = id2name.get(history_ids[-1], "")
    last_meta = exercise_meta_dict.get(last_ex_name, {})

    for s, idx in zip(scores.cpu().numpy(), indices.cpu().numpy()):
        idx_val = int(idx)
        name = id2name[idx_val]
        current_meta = exercise_meta_dict.get(name, {})
        is_same = 1 if current_meta.get('category') == last_meta.get('category') else 0
        
        raw_eq = id_to_eq_dict.get(idx_val, 'Unknown')
        # Маппим инвентарь упражнения к классам обучения (Bands/Cardio -> Bodyweight)
        mapped_eq = INVENTORY_MAP.get(raw_eq, raw_eq)
        
        candidates.append({
            'bert_score': float(s),
            'candidate_id': str(idx_val),
            'is_same_group': is_same,
            'eq_clean': str(mapped_eq),
            'level_idx': LEVEL_MAP.get(user_profile['level'], 1),
            'goal_clean': str(user_profile['goal']),
            'raw_equipment': raw_eq 
        })
    
    cand_df = pd.DataFrame(candidates)
    features_order = ['bert_score', 'candidate_id', 'is_same_group', 'eq_clean', 'level_idx', 'goal_clean']
    
    preds = cat_ranker.predict_proba(cand_df[features_order])[:, 1]
    cand_df['final_score'] = preds
    cand_df['name'] = [id2name[int(i)] for i in cand_df['candidate_id']]
    
    # --- ХАРД-ФИЛЬТР ---
    user_selection = user_profile['equipment']
    if user_selection != 'All (Gym Mixed)':
        # Если юзер выбрал Bodyweight, фильтр пропустит всё, что замаплено в Bodyweight (включая Cardio/Bands)
        cand_df = cand_df[cand_df['eq_clean'] == user_selection]
    
    final_df = cand_df.sort_values('final_score', ascending=False).head(top_k)
    
    cat_table = final_df[['name', 'final_score', 'bert_score', 'raw_equipment']].copy()
    cat_table.columns = ['Упражнение', 'CatBoost Prob', 'BERT Score', 'Инвентарь']
    
    bert_df = pd.DataFrame({
        "Упражнение": [id2name[i.item()] for i in indices[:10]],
        "BERT Score": [round(s.item(), 4) for s in scores[:10]]
    })
    
    return final_df['name'].tolist(), bert_df, cat_table

# --- 4. ИНТЕРФЕЙС GRADIO ---
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    history_state = gr.State([])
    profile_state = gr.State({})
    
    gr.Markdown("# 🏋️‍♂️ AI Trainer: Final Reranker Test (v3)")
    
    with gr.Row(variant='panel'):
        level = gr.Dropdown(['Novice', 'Beginner', 'Intermediate', 'Advanced'], label="Уровень", value="Intermediate")
        goal = gr.Dropdown(['Powerbuilding', 'Bodybuilding', 'Athletics', 'Powerlifting', 'Muscle & Sculpting', 'Bodyweight Fitness', 'Fitness'], label="Цель", value="Bodybuilding")
        # Убрали Cardio и Bands из выбора юзера, оставив общую категорию Bodyweight
        equip = gr.Dropdown(['Machine', 'Dumbbell', 'Barbell', 'Bodyweight', 'All (Gym Mixed)'], label="Инвентарь", value="All (Gym Mixed)")
        start_btn = gr.Button("🚀 Настроить профиль", variant="primary")

    with gr.Column(visible=False) as main_app:
        history_display = gr.Textbox(label="Цепочка тренировки", interactive=False)
        
        with gr.Row() as first_ex_row:
            first_ex_drop = gr.Dropdown(choices=sorted(list(vocab.keys())), label="Выберите первое упражнение:", interactive=True)
            
        with gr.Row(visible=False) as results:
            with gr.Column():
                gr.Markdown("#### 🧠 BERT (Raw)")
                bert_ui_table = gr.DataFrame()
            with gr.Column():
                gr.Markdown("#### ⚖️ CatBoost (Reranked)")
                cat_ui_table = gr.DataFrame()
        
        recommendations = gr.Dropdown(choices=[], label="Что дальше?", interactive=True, visible=False)
        add_btn = gr.Button("➕ Добавить в цепочку", variant="primary", visible=False)
        reset_btn = gr.Button("🔄 Сброс", variant="stop")

    def start_fn(l, g, e):
        return {profile_state: {'level': l, 'goal': g, 'equipment': e}, main_app: gr.update(visible=True)}

    def add_ex_fn(ex_name, history, profile):
        if not ex_name: return history, "", gr.update(), None, None, gr.update(), gr.update(), gr.update()
        new_id = vocab[ex_name]
        if not history or history[-1] != new_id: history.append(new_id)
        
        recs, b_df, c_df = predict_debug(history, profile)
        return (history, " ➔ ".join([id2name[i] for i in history]), 
                gr.update(choices=recs, value=None, visible=True), 
                b_df, c_df, gr.update(visible=True), gr.update(visible=True), gr.update(visible=False))

    start_btn.click(start_fn, [level, goal, equip], [profile_state, main_app])
    first_ex_drop.change(add_ex_fn, [first_ex_drop, history_state, profile_state], 
                         [history_state, history_display, recommendations, bert_ui_table, cat_ui_table, results, add_btn, first_ex_row])
    add_btn.click(add_ex_fn, [recommendations, history_state, profile_state], 
                  [history_state, history_display, recommendations, bert_ui_table, cat_ui_table, results, add_btn, first_ex_row])
    reset_btn.click(lambda: ([], "", gr.update(visible=False), gr.update(visible=True)), None, [history_state, history_display, main_app, first_ex_row])

demo.launch(debug=True, share=True)

/Users/artemmarkov/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/artemmarkov/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ CatBoost загружен: ../gym_model/catboost_recommender_final.cbm
Running on local URL:  http://127.0.0.1:7860
IMPORTANT: You are using gradio version 3.41.2, however version 4.44.1 is available, please upgrade.
--------
Running on public URL: https://33522410caaad1bec2.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://33522410caaad1bec2.gradio.live
